In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!apt-get update
!apt-get install -y tree


Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,297 kB]
Get:6 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,607 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,289 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy

In [ ]:
!ls /content/drive/MyDrive/



 acrobat			     Screenshot_20230908-123404.png
'adobe reader manual'		     Udaipur.gdoc
'Camera Roll'			    'Untitled document (1).gdoc'
 certificate.pdf		    'Untitled document (2).gdoc'
 Classroom			    'Untitled document (3).gdoc'
'Colab Notebooks'		    'Untitled document (4).gdoc'
'Contact Information (1).gform'     'Untitled document.gdoc'
'Contact Information.gform'	    'Untitled Jam.pdf'
 dataset			    'vinay docs'
'Google AI Studio'		     VinayPatil_OyrtaContentAssignment.pdf
 IMG_20211101_155543.jpg	    'Vinay Patil resume.docx'
 IMG_20211101_155601.jpg	    'vinay resume (1).pdf'
 IMG_20211101_155639.jpg	    'Vinayresume (1).pdf'
 IMG_20211101_161304.jpg	    'Vinayresume (2).pdf'
 IMG_20231126_194158_818.jpg	    'vinay resume.pdf'
 INTERIOR.gdoc			     Vinayresume.pdf
'My certificates'		    'vinays resume  (1).pdf'
'New folder'			    'vinays resume .pdf'
 Picsart_22-07-25_20-10-56-998.png   Wait.gdoc


In [ ]:
!ls /content/drive/MyDrive/dataset/urban_issue.zip



/content/drive/MyDrive/dataset/urban_issue.zip


In [ ]:
!unzip /content/drive/MyDrive/dataset/urban_issue.zip -d /content/urban_issue


Streaming output truncated to the last 5000 lines.
  inflating: /content/urban_issue/urban issue/Potholes and RoadCracks/Potholes and RoadCracks/train/images/387_jpg.rf.4134302ff330ff2ffb62d82d606c151e.jpg  
  inflating: /content/urban_issue/urban issue/Potholes and RoadCracks/Potholes and RoadCracks/train/images/387_jpg.rf.458714070c61b9628f3b2540f2dcc5a2.jpg  
  inflating: /content/urban_issue/urban issue/Potholes and RoadCracks/Potholes and RoadCracks/train/images/387_jpg.rf.5e46d9121b5aa2e54db60b62ff2d8e94.jpg  
  inflating: /content/urban_issue/urban issue/Potholes and RoadCracks/Potholes and RoadCracks/train/images/387_jpg.rf.87db4fec1ab3d8f5270fdb3e5e54b63f.jpg  
  inflating: /content/urban_issue/urban issue/Potholes and RoadCracks/Potholes and RoadCracks/train/images/387_jpg.rf.8cdad1e8540439ee255b1f24a2cebcd4.jpg  
  inflating: /content/urban_issue/urban issue/Potholes and RoadCracks/Potholes and RoadCracks/train/images/387_jpg.rf.b6e0f7585485bf68a3021ff6a07961f6.jpg  
  infla

In [ ]:
!tree -d -L 3 /content/urban_issue


/bin/bash: line 1: tree: command not found


In [ ]:
import os
import shutil
import random

SOURCE_ROOT = "/content/urban_issue/urban issue"
DEST_ROOT = "/content/urban_issue_final"

SPLITS = {"train": 0.8, "val": 0.1, "test": 0.1}
random.seed(42)

def safe_name(name):
    return (
        name.lower()
        .replace(" ", "_")
        .replace("&", "and")
        .replace("__", "_")
    )

def copy_files(files, dest):
    os.makedirs(dest, exist_ok=True)
    for f in files:
        shutil.copy(f, dest)

for issue in os.listdir(SOURCE_ROOT):
    issue_path = os.path.join(SOURCE_ROOT, issue)
    if not os.path.isdir(issue_path):
        continue

    class_name = safe_name(issue)
    print(f"Processing: {class_name}")

    # Case: already has Train/Test/Validation
    subdirs = [d.lower() for d in os.listdir(issue_path)]
    if {"train", "test", "validation"}.issubset(set(subdirs)):
        for split in ["Train", "Test", "Validation"]:
            src = os.path.join(issue_path, split)
            dst = os.path.join(
                DEST_ROOT,
                "val" if split == "Validation" else split.lower(),
                class_name
            )
            os.makedirs(dst, exist_ok=True)

            for root, _, files in os.walk(src):
                for file in files:
                    shutil.copy(os.path.join(root, file), dst)

    else:
        # Collect images (handle nested duplicate folders)
        images = []
        for root, _, files in os.walk(issue_path):
            for f in files:
                if f.lower().endswith((".jpg", ".png", ".jpeg")):
                    images.append(os.path.join(root, f))

        random.shuffle(images)

        n = len(images)
        t = int(n * SPLITS["train"])
        v = int(n * SPLITS["val"])

        split_map = {
            "train": images[:t],
            "val": images[t:t+v],
            "test": images[t+v:]
        }

        for split, files in split_map.items():
            dest = os.path.join(DEST_ROOT, split, class_name)
            copy_files(files, dest)

print("✅ Dataset restructuring completed.")


Processing: graffitti
Processing: deadanimalspollution
Processing: damagedroadsigns
Processing: fallentrees
Processing: potholes_and_roadcracks
Processing: damagedelectricalpoles
Processing: footpath_split
Processing: damaged_concrete_structures
Processing: illegalparking
Processing: garbage
✅ Dataset restructuring completed.


In [ ]:
!tree -d -L 3 /content/urban_issue_final


/content/urban_issue_final
├── test
│   ├── damaged_concrete_structures
│   ├── damagedelectricalpoles
│   ├── damagedroadsigns
│   ├── deadanimalspollution
│   ├── fallentrees
│   ├── footpath_split
│   ├── garbage
│   ├── graffitti
│   ├── illegalparking
│   └── potholes_and_roadcracks
├── train
│   ├── damaged_concrete_structures
│   ├── damagedelectricalpoles
│   ├── damagedroadsigns
│   ├── deadanimalspollution
│   ├── fallentrees
│   ├── footpath_split
│   ├── garbage
│   ├── graffitti
│   ├── illegalparking
│   └── potholes_and_roadcracks
└── val
    ├── damaged_concrete_structures
    ├── damagedelectricalpoles
    ├── damagedroadsigns
    ├── deadanimalspollution
    ├── fallentrees
    ├── footpath_split
    ├── garbage
    ├── graffitti
    ├── illegalparking
    └── potholes_and_roadcracks

33 directories


In [ ]:
from PIL import Image

def remove_corrupt_images(path):
    bad = 0
    for root, _, files in os.walk(path):
        for f in files:
            try:
                Image.open(os.path.join(root, f)).verify()
            except:
                os.remove(os.path.join(root, f))
                bad += 1
    print("Removed corrupt images:", bad)

remove_corrupt_images("/content/urban_issue_final")


Removed corrupt images: 0


In [ ]:
# import Libraries
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os


In [ ]:
# ================= CONFIG =================
DATA_DIR = "/content/urban_issue_final"

BATCH_SIZE = 32      # You can change to 16 if GPU memory is low
IMG_SIZE = 224
EPOCHS = 10
LR = 1e-4
NUM_WORKERS = 2
# =========================================


In [ ]:
data_transforms = {
    "train": transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ]),
    "val": transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ]),
    "test": transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
}


In [ ]:


image_datasets = {
    x: datasets.ImageFolder(
        root=os.path.join(DATA_DIR, x),
        transform=data_transforms[x]
    )
    for x in ["train", "val", "test"]
}

dataloaders = {
    x: DataLoader(
        image_datasets[x],
        batch_size=BATCH_SIZE,
        shuffle=True if x == "train" else False,
        num_workers=2
    )
    for x in ["train", "val", "test"]
}

class_names = image_datasets["train"].classes
num_classes = len(class_names)

print("Classes:", class_names)
print("Number of classes:", num_classes)


Classes: ['damaged_concrete_structures', 'damagedelectricalpoles', 'damagedroadsigns', 'deadanimalspollution', 'fallentrees', 'footpath_split', 'garbage', 'graffitti', 'illegalparking', 'potholes_and_roadcracks']
Number of classes: 10


In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader


In [ ]:
data_transforms = {
    "train": transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ]),
    "val": transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ]),
    "test": transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
}


In [ ]:
image_datasets = {
    x: datasets.ImageFolder(
        root=os.path.join(DATA_DIR, x),
        transform=data_transforms[x]
    )
    for x in ["train", "val", "test"]
}

dataloaders = {
    x: DataLoader(
        image_datasets[x],
        batch_size=BATCH_SIZE,
        shuffle=True if x == "train" else False,
        num_workers=NUM_WORKERS
    )
    for x in ["train", "val", "test"]
}

class_names = image_datasets["train"].classes
num_classes = len(class_names)

print("Classes:", class_names)
print("Total classes:", num_classes)


Classes: ['damaged_concrete_structures', 'damagedelectricalpoles', 'damagedroadsigns', 'deadanimalspollution', 'fallentrees', 'footpath_split', 'garbage', 'graffitti', 'illegalparking', 'potholes_and_roadcracks']
Total classes: 10


In [ ]:
import numpy as np
from collections import Counter

targets = image_datasets["train"].targets
class_counts = Counter(targets)

num_classes = len(class_counts)
total_samples = sum(class_counts.values())

class_weights = [
    total_samples / class_counts[i]
    for i in range(num_classes)
]

class_weights = torch.tensor(class_weights).to(DEVICE)

print("Class counts:", class_counts)
print("Class weights:", class_weights)


Class counts: Counter({0: 9412, 4: 8891, 1: 6481, 9: 5804, 6: 3304, 2: 1879, 7: 1702, 5: 990, 3: 176, 8: 53})
Class weights: tensor([  4.1109,   5.9701,  20.5918, 219.8409,   4.3518,  39.0828,  11.7107,
         22.7333, 730.0377,   6.6664], device='cuda:0')


In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

# Freeze backbone
for param in model.parameters():
    param.requires_grad = False

# Replace classifier
model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(DEVICE)


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 189MB/s]


In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.Adam(model.fc.parameters(), lr=LR)


In [ ]:
def train_model(model, epochs):
    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        print("-" * 40)

        for phase in ["train", "val"]:
            model.train() if phase == "train" else model.eval()

            running_loss = 0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels)

            epoch_loss = running_loss / len(image_datasets[phase])
            epoch_acc = running_corrects.double() / len(image_datasets[phase])

            print(f"{phase.upper()} | Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}")

    return model


In [ ]:
model = train_model(model, EPOCHS)



Epoch 1/10
----------------------------------------
TRAIN | Loss: 0.8111 | Acc: 0.8536
VAL | Loss: 0.6293 | Acc: 0.8886

Epoch 2/10
----------------------------------------
TRAIN | Loss: 0.5397 | Acc: 0.8825
VAL | Loss: 0.4375 | Acc: 0.9045

Epoch 3/10
----------------------------------------
TRAIN | Loss: 0.4253 | Acc: 0.8973
VAL | Loss: 0.3643 | Acc: 0.9143

Epoch 4/10
----------------------------------------
TRAIN | Loss: 0.3573 | Acc: 0.9081
VAL | Loss: 0.3403 | Acc: 0.9243

Epoch 5/10
----------------------------------------
TRAIN | Loss: 0.3106 | Acc: 0.9158
VAL | Loss: 0.2812 | Acc: 0.9277

Epoch 6/10
----------------------------------------
TRAIN | Loss: 0.2787 | Acc: 0.9221
VAL | Loss: 0.2460 | Acc: 0.9358

Epoch 7/10
----------------------------------------
TRAIN | Loss: 0.2566 | Acc: 0.9268
VAL | Loss: 0.2335 | Acc: 0.9379

Epoch 8/10
----------------------------------------
TRAIN | Loss: 0.2379 | Acc: 0.9308
VAL | Loss: 0.2170 | Acc: 0.9390

Epoch 9/10
--------------------

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

y_true = []
y_pred = []

model.eval()
with torch.no_grad():
    for inputs, labels in dataloaders["test"]:
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())


In [ ]:
cm = confusion_matrix(y_true, y_pred)
print("Confusion Matrix:\n", cm)


Confusion Matrix:
 [[1080    2    1    1    5    2   12   60    1   13]
 [   4  764    4    0   16    3    8    9    1    2]
 [   0    0  234    0    0    0    0    2    0    0]
 [   0    0    0   22    0    0    0    0    0    0]
 [   1   30    1    1 1057    5   12    5    0    0]
 [   2    0    0    0    0  167    2    1    0    2]
 [   1    3    0    2    2    2  391   10    0    2]
 [   4    2    3    0    2    1    2  200    0    0]
 [   0    0    0    0    0    0    0    1    7    0]
 [   3    1    0    0    2    3    0    8    3  706]]


In [ ]:
print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names
    )
)


                             precision    recall  f1-score   support

damaged_concrete_structures       0.99      0.92      0.95      1177
     damagedelectricalpoles       0.95      0.94      0.95       811
           damagedroadsigns       0.96      0.99      0.98       236
       deadanimalspollution       0.85      1.00      0.92        22
                fallentrees       0.98      0.95      0.96      1112
             footpath_split       0.91      0.96      0.94       174
                    garbage       0.92      0.95      0.93       413
                  graffitti       0.68      0.93      0.78       214
             illegalparking       0.58      0.88      0.70         8
    potholes_and_roadcracks       0.97      0.97      0.97       726

                   accuracy                           0.95      4893
                  macro avg       0.88      0.95      0.91      4893
               weighted avg       0.95      0.95      0.95      4893



In [ ]:
torch.save({
    "model_state": model.state_dict(),
    "class_names": class_names
}, "urban_issue_resnet50.pth")


In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load checkpoint
checkpoint = torch.load("urban_issue_resnet50.pth", map_location=DEVICE)

class_names = checkpoint["class_names"]
num_classes = len(class_names)

# Recreate model architecture
model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, num_classes)

model.load_state_dict(checkpoint["model_state"])
model = model.to(DEVICE)
model.eval()

print("✅ Model loaded successfully")
print("Classes:", class_names)


✅ Model loaded successfully
Classes: ['damaged_concrete_structures', 'damagedelectricalpoles', 'damagedroadsigns', 'deadanimalspollution', 'fallentrees', 'footpath_split', 'garbage', 'graffitti', 'illegalparking', 'potholes_and_roadcracks']


In [ ]:
IMG_SIZE = 224

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [ ]:
import torch.nn.functional as F

def predict_image(image_path, model, class_names):
    image = Image.open(image_path).convert("RGB")
    image = test_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(image)
        probs = F.softmax(outputs, dim=1)

        confidence, pred_idx = torch.max(probs, dim=1)

    predicted_class = class_names[pred_idx.item()]
    confidence_score = confidence.item()

    return predicted_class, confidence_score


In [ ]:

image_path = "/content/download1.jpeg"

pred_class, confidence = predict_image(image_path, model, class_names)

print("🖼 Image:", image_path)
print("✅ Predicted Issue:", pred_class)
print("🎯 Confidence Score:", round(confidence * 100, 2), "%")


🖼 Image: /content/download1.jpeg
✅ Predicted Issue: potholes_and_roadcracks
🎯 Confidence Score: 68.72 %


In [ ]:
import h5py


In [ ]:
import h5py

def save_model_to_h5(model, filepath):
    with h5py.File(filepath, "w") as h5f:
        for key, value in model.state_dict().items():
            h5f.create_dataset(key, data=value.cpu().numpy())

    print(f"✅ Model saved in H5 format at {filepath}")


In [ ]:
save_model_to_h5(model, "urban_issue_resnet50.h5")


✅ Model saved in H5 format at urban_issue_resnet50.h5


In [ ]:
!mkdir -p /content/drive/MyDrive/checkpoints


In [ ]:
!ls /content/drive/MyDrive



 acrobat			  Picsart_22-07-25_20-10-56-998.png
'adobe reader manual'		  Screenshot_20230908-123404.png
'Camera Roll'			  Udaipur.gdoc
 certificate.pdf		 'Untitled document (1).gdoc'
 checkpoints			 'Untitled document (2).gdoc'
 Classroom			 'Untitled document (3).gdoc'
'Colab Notebooks'		 'Untitled document (4).gdoc'
'Contact Information (1).gform'  'Untitled document.gdoc'
'Contact Information.gform'	 'Untitled Jam.pdf'
 dataset			 'vinay docs'
'Google AI Studio'		  VinayPatil_OyrtaContentAssignment.pdf
 IMG_20211101_155543.jpg	 'Vinay Patil resume.docx'
 IMG_20211101_155601.jpg	 'vinay resume (1).pdf'
 IMG_20211101_155639.jpg	 'Vinayresume (1).pdf'
 IMG_20211101_161304.jpg	 'Vinayresume (2).pdf'
 IMG_20231126_194158_818.jpg	 'vinay resume.pdf'
 INTERIOR.gdoc			  Vinayresume.pdf
'My certificates'		 'vinays resume .pdf'
'New folder'			  Wait.gdoc


In [ ]:
CHECKPOINT_PATH = "/content/drive/MyDrive/checkpoints/urban_issue_checkpoint.pth"
print("Checkpoint will be saved at:", CHECKPOINT_PATH)



Checkpoint will be saved at: /content/drive/MyDrive/checkpoints/urban_issue_checkpoint.pth


In [ ]:
def save_checkpoint(model, optimizer, epoch, path):
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "class_names": class_names
    }, path)


In [ ]:
CHECKPOINT_PATH = "/content/drive/MyDrive/urban_issue_checkpoint.pth"

def train_model(model, epochs, start_epoch=0):
    for epoch in range(start_epoch, epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        print("-" * 40)

        for phase in ["train", "val"]:
            model.train() if phase == "train" else model.eval()

            running_loss = 0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == "train"):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels)

            epoch_loss = running_loss / len(image_datasets[phase])
            epoch_acc = running_corrects.double() / len(image_datasets[phase])

            print(f"{phase.upper()} | Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.4f}")

        # 🔥 SAVE AFTER EVERY EPOCH
        save_checkpoint(model, optimizer, epoch + 1, CHECKPOINT_PATH)
        print(f"💾 Checkpoint saved (Epoch {epoch+1})")

    return model


In [ ]:
save_checkpoint(model, optimizer, 0, CHECKPOINT_PATH)


In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(DEVICE)

optimizer = torch.optim.Adam(model.fc.parameters(), lr=LR)


In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

model.load_state_dict(checkpoint["model_state"])
optimizer.load_state_dict(checkpoint["optimizer_state"])
start_epoch = checkpoint["epoch"]

class_names = checkpoint["class_names"]

print(f"✅ Resuming training from Epoch {start_epoch}")


✅ Resuming training from Epoch 0


In [ ]:
model = train_model(model, EPOCHS, start_epoch=start_epoch)



Epoch 1/10
----------------------------------------
TRAIN | Loss: 1.2911 | Acc: 0.7854
VAL | Loss: 0.7594 | Acc: 0.8576
💾 Checkpoint saved (Epoch 1)

Epoch 2/10
----------------------------------------
TRAIN | Loss: 0.6581 | Acc: 0.8676
VAL | Loss: 0.5261 | Acc: 0.8892
💾 Checkpoint saved (Epoch 2)

Epoch 3/10
----------------------------------------
TRAIN | Loss: 0.4794 | Acc: 0.8879
VAL | Loss: 0.4288 | Acc: 0.9153
💾 Checkpoint saved (Epoch 3)

Epoch 4/10
----------------------------------------
TRAIN | Loss: 0.3897 | Acc: 0.9017
VAL | Loss: 0.3347 | Acc: 0.9220
💾 Checkpoint saved (Epoch 4)

Epoch 5/10
----------------------------------------
TRAIN | Loss: 0.3386 | Acc: 0.9130
VAL | Loss: 0.2957 | Acc: 0.9220
💾 Checkpoint saved (Epoch 5)

Epoch 6/10
----------------------------------------


KeyboardInterrupt: 

In [ ]:
model.eval()
correct, total = 0, 0

with torch.no_grad():
    for images, labels in dataloaders["test"]:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

print("✅ TEST Accuracy:", correct / total)


✅ TEST Accuracy: 0.9282648681790313


In [ ]:
torch.save({
    "model_state": model.state_dict(),
    "class_names": class_names
}, "/content/drive/MyDrive/checkpoints/urban_issue_resnet50_final.pth")
